# <center> Homework 111

In [23]:
import tensorflow as tf
import keras_tuner as kt
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from importlib import reload
import tf_model 
import joblib

## Task 0

In [24]:
X, y = fetch_california_housing(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42, test_size=0.2, shuffle=True) 
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, random_state=42, test_size=0.1, shuffle=True) 

In [ ]:
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < threshold
        squared_loss = tf.square(error) / 2
        linear_loss = threshold * tf.abs(error) - threshold ** 2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    return huber_fn

In [6]:
norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

model_0 = tf.keras.models.Sequential([
    tf.keras.layers.Input(X_train.shape[1:]),
    norm,
    tf.keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(1)
])

model_0.compile(loss=create_huber(2.0), optimizer="nadam", metrics=['RootMeanSquaredError'])
model_0.fit(X_train, y_train, epochs=20, validation_data=(X_val, y_val))

2026-01-19 20:16:43.269578: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - RootMeanSquaredError: 1.1323 - loss: 0.3191 - val_RootMeanSquaredError: 1.0377 - val_loss: 0.3026
Epoch 2/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 0.9958 - loss: 0.2249 - val_RootMeanSquaredError: 0.7110 - val_loss: 0.2245
Epoch 3/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.7865 - loss: 0.2037 - val_RootMeanSquaredError: 0.6580 - val_loss: 0.1967
Epoch 4/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 0.6893 - loss: 0.1865 - val_RootMeanSquaredError: 0.6025 - val_loss: 0.1780
Epoch 5/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.8288 - loss: 0.1854 - val_RootMeanSquaredError: 0.6122 - val_loss: 0.1843
Epoch 6/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - RootMeanSquaredError: 0.6716 - loss: 0.1718 - val_RootMeanSquaredError: 0.6314 - val_loss: 0.1897
Epoch 7/20
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.6536 - los

In [7]:
model_0.save('model_0_111.keras')

In [9]:
loaded_model_0 = tf.keras.models.load_model('model_0_111.keras', 
                                            custom_objects={'huber_fn': create_huber(2.0)})

loaded_model_0.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - RootMeanSquaredError: 0.5101 - loss: 0.1284 - val_RootMeanSquaredError: 0.5742 - val_loss: 0.1618
Epoch 2/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.5149 - loss: 0.1285 - val_RootMeanSquaredError: 0.5641 - val_loss: 0.1559
Epoch 3/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - RootMeanSquaredError: 0.5042 - loss: 0.1256 - val_RootMeanSquaredError: 0.5693 - val_loss: 0.1586
Epoch 4/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.5085 - loss: 0.1261 - val_RootMeanSquaredError: 0.5614 - val_loss: 0.1547
Epoch 5/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - RootMeanSquaredError: 0.5051 - loss: 0.1243 - val_RootMeanSquaredError: 0.6015 - val_loss: 0.1720
Epoch 6/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 0.4998 - loss: 0.1232 - val_RootMeanSquaredError: 0.5627 - val_loss: 0.1553
Epoch 7/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 0.4941 - los

In [25]:
class HuberLoss(tf.keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss = self.threshold * tf.abs(error) - self.threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

In [28]:
norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

model_01 = tf.keras.models.Sequential([
    tf.keras.layers.Input(X_train.shape[1:]),
    norm,
    tf.keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(1)
])

model_01.compile(loss=HuberLoss(2.), optimizer="nadam", metrics=['RootMeanSquaredError'])
model_01.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - RootMeanSquaredError: 1.6882 - loss: 0.7690 - val_RootMeanSquaredError: 1.1694 - val_loss: 0.6977
Epoch 2/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - RootMeanSquaredError: 1.6801 - loss: 0.7116 - val_RootMeanSquaredError: 1.2130 - val_loss: 0.7126
Epoch 3/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - RootMeanSquaredError: 1.3444 - loss: 0.6872 - val_RootMeanSquaredError: 1.2049 - val_loss: 0.7073
Epoch 4/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 1.2778 - loss: 0.6772 - val_RootMeanSquaredError: 1.1665 - val_loss: 0.6773
Epoch 5/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 1.1666 - loss: 0.6538 - val_RootMeanSquaredError: 1.1585 - val_loss: 0.6754
Epoch 6/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - RootMeanSquaredError: 1.2007 - loss: 0.6673 - val_RootMeanSquaredError: 1.1487 - val_loss: 0.6765
Epoch 7/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - RootMeanSquaredError: 1.1823 - los

In [29]:
model_01.save('model_01.keras')

In [30]:
loaded_model_01 = tf.keras.models.load_model('model_01.keras', 
                                             custom_objects={"HuberLoss": HuberLoss})

loaded_model_01.fit(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

Epoch 1/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - RootMeanSquaredError: 1.1330 - loss: 0.6455 - val_RootMeanSquaredError: 1.1499 - val_loss: 0.6733
Epoch 2/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 1.1500 - loss: 0.6478 - val_RootMeanSquaredError: 1.1547 - val_loss: 0.6816
Epoch 3/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 1.1349 - loss: 0.6453 - val_RootMeanSquaredError: 1.1822 - val_loss: 0.6736
Epoch 4/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 1.1369 - loss: 0.6437 - val_RootMeanSquaredError: 1.1268 - val_loss: 0.6749
Epoch 5/5
465/465 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - RootMeanSquaredError: 1.1304 - loss: 0.6442 - val_RootMeanSquaredError: 1.1685 - val_loss: 0.6772


## Task 2

In [11]:
reload(tf_model)
from tf_model import *

norm = Normalization()
norm.adapt(X_train)

model_2 = Sequential([
    Input(X_train.shape[1:]),
    norm,
    Dense(300, activation='relu', kernel_initializer='he_normal'),
    Dense(100, activation='relu', kernel_initializer='he_normal'),
    Dense(1)
])

model_2.compile(loss='mse', optimizer=Adam(), metrics=['rmse'])
model_2.fit(X_train, y_train, epochs=20, validation_data=(X_val, y_val))

Epoch 1/20


100%|██████████| 465/465 [00:23<00:00, 20.08it/s]


    - loss: 1.7491612434387207 - rmse: 4.742878437042236
    - val_loss: 0.4856973886489868 - val_rmse: 0.6969199202623162
Learning Rate: 0.001
Epoch 2/20


100%|██████████| 465/465 [00:20<00:00, 23.14it/s]


    - loss: 0.6291747093200684 - rmse: 0.6890851855278015
    - val_loss: 0.41406646370887756 - val_rmse: 0.6434799547050842
Learning Rate: 0.001
Epoch 3/20


...

    - loss: 0.33877891302108765 - rmse: 0.42077329754829407
    - val_loss: 0.3253108859062195 - val_rmse: 0.5703603239194852
Learning Rate: 0.001
Epoch 19/20


100%|██████████| 465/465 [00:19<00:00, 23.73it/s]


    - loss: 0.29816365242004395 - rmse: 0.5364454984664917
    - val_loss: 0.337129682302475 - val_rmse: 0.5806286995263609
Learning Rate: 0.001
Epoch 20/20


100%|██████████| 465/465 [00:19<00:00, 23.95it/s]

    - loss: 0.2695308327674866 - rmse: 0.745652437210083
    - val_loss: 0.32283496856689453 - val_rmse: 0.5681857014097296
Learning Rate: 0.001


In [14]:
model_2.predict(X_test[:3])

<tf.Tensor: shape=(3, 1), dtype=float32, numpy=
array([[0.17978805],
       [1.1233771 ],
       [4.526843  ]], dtype=float32)>

In [12]:
model_2.save('custom_model_2.pkl')

In [13]:
loaded_model_2 = load_model('custom_model_2.pkl')
loaded_model_2.predict(X_test[:3])

<tf.Tensor: shape=(3, 1), dtype=float32, numpy=
array([[0.17978805],
       [1.1233771 ],
       [4.526843  ]], dtype=float32)>

In [16]:
loaded_model_2.fit(X_train, y_train, epochs=5, validation_data=(X_val, y_val))

Epoch 1/5


100%|██████████| 465/465 [00:21<00:00, 21.21it/s]


    - loss: 0.24494381248950958 - rmse: 0.7043812274932861
    - val_loss: 0.30938345193862915 - val_rmse: 0.5562224741472048
Learning Rate: 0.001
Epoch 2/5


100%|██████████| 465/465 [00:19<00:00, 23.27it/s]


    - loss: 0.2615848183631897 - rmse: 0.8719172477722168
    - val_loss: 0.31722408533096313 - val_rmse: 0.5632264981280504
Learning Rate: 0.001
Epoch 3/5


...

    - loss: 0.26072776317596436 - rmse: 0.693024754524231
    - val_loss: 0.32045844197273254 - val_rmse: 0.5660904707977009
Learning Rate: 0.001
Epoch 4/5


100%|██████████| 465/465 [00:17<00:00, 27.18it/s]


    - loss: 0.28282430768013 - rmse: 0.3473430573940277
    - val_loss: 0.3179105818271637 - val_rmse: 0.5638355927760039
Learning Rate: 0.001
Epoch 5/5


100%|██████████| 465/465 [00:14<00:00, 32.01it/s]

    - loss: 0.3048596680164337 - rmse: 0.6001268625259399
    - val_loss: 0.31604915857315063 - val_rmse: 0.5621824947504197
Learning Rate: 0.001


In [9]:
reload(tf_model)

<module 'tf_model' from '/home/zdravko/Machine_Learning_Intern/Machine_Learning/2026/tf_model.py'>

In [10]:

loaded_model_2 = joblib.load('custom_model_2.pkl')
loaded_model_2.save('test_save')

da


In [17]:
reload(tf_model)
from tf_model import *

l_loaded_model_2 = load_model('test_save')

In [18]:
l_loaded_model_2.predict(X_test[:3])

<tf.Tensor: shape=(3, 1), dtype=float32, numpy=
array([[-0.17982873],
       [ 0.01586491],
       [-0.30450547]], dtype=float32)>

In [22]:
l_loaded_model_2.fit(X_train, y_train, epochs=2, validation_data=(X_val, y_val))

Epoch 1/2


100%|██████████| 465/465 [00:22<00:00, 20.73it/s]


    - loss: 3.4663803577423096 - rmse: 3.0113887786865234
    - val_loss: 0.4757603108882904 - val_rmse: 0.6897538283604194
Learning Rate: 0.001
Epoch 2/2


100%|██████████| 465/465 [00:22<00:00, 21.01it/s]

    - loss: 0.5882796049118042 - rmse: 0.6627270579338074
    - val_loss: 0.4039594829082489 - val_rmse: 0.6355780773700724
Learning Rate: 0.001


## Task 3

In [20]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [23]:
def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=5, max_value=8)
    n_neurons = hp.Int("n_neurons", min_value=128, max_value=256)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    optimizer_choice = hp.Choice('optimizer', ['sgd', 'adam', 'nadam', 'adamax'])
    match optimizer_choice:
      case 'nadam': optimizer = tf.keras.optimizers.Nadam(learning_rate)
      case 'adam': optimizer = tf.keras.optimizers.Adam(learning_rate)
      case 'adamax': optimizer = tf.keras.optimizers.Adamax(learning_rate)
      case 'sgd': optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                        momentum=hp.Choice("momentum", [0.0, 0.95]),
                                        nesterov=hp.Boolean("nesterov"))

    
    activation = hp.Choice("activation", values=["swish", "gelu", 'elu', 'relu', 'leaky_relu'])
    batch_norm = hp.Boolean('BN')

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(X_train.shape[1:]))
    model.add(tf.keras.layers.Flatten())

    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, kernel_initializer='he_normal'))
        if batch_norm:
            model.add(tf.keras.layers.BatchNormalization())

        if activation == 'leaky_relu':
            layer = tf.keras.layers.LeakyReLU()
        else:
            layer = tf.keras.layers.Activation(activation)

        model.add(layer)
    
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

In [24]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10,
    directory="fashion_mnist",
    project_name="params_2_111",
    seed=42,
)

tuner.search(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=5,
    batch_size=64,
)

Trial 10 Complete [00h 01m 16s]
val_accuracy: 0.8866000175476074

Best val_accuracy So Far: 0.8866000175476074
Total elapsed time: 00h 12m 39s


In [25]:
best_hp = tuner.get_best_hyperparameters(1)[0]
best_hp.values

{'n_hidden': 5,
 'n_neurons': 243,
 'learning_rate': 0.0043925333029596605,
 'optimizer': 'sgd',
 'momentum': 0.95,
 'nesterov': True,
 'activation': 'elu',
 'BN': False}

In [26]:
best_model_3 = tuner.get_best_models(1)[0]

callbacks = [tf.keras.callbacks.EarlyStopping(patience=8)]
best_model_3.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid), callbacks=callbacks)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.8804 - loss: 0.3204 - val_accuracy: 0.8642 - val_loss: 0.3716
Epoch 2/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.8893 - loss: 0.2942 - val_accuracy: 0.8692 - val_loss: 0.3602
Epoch 3/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.8966 - loss: 0.2737 - val_accuracy: 0.8736 - val_loss: 0.3634
Epoch 4/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9036 - loss: 0.2594 - val_accuracy: 0.8758 - val_loss: 0.3625
Epoch 5/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9076 - loss: 0.2459 - val_accuracy: 0.8788 - val_loss: 0.3585
Epoch 6/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - accuracy: 0.9113 - loss: 0.2366 - val_accuracy: 0.8778 - val_loss: 0.3546
Epoch 7/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - accuracy: 0.9150 - loss: 0.2265 - val_accuracy: 0.8822 - val_loss: 0.3515
Epoch 8/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9168 - los

In [27]:
best_model_3.save('best_model_3_111.keras')

In [28]:
best_model_3.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 243)            │       190,755 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 243)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 243)            │        59,292 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 243)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 243)            │        59,292 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 243)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 243)            │        59,292 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 243)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 243)            │        59,292 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 243)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         2,440 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 860,728 (3.28 MB)

 Trainable params: 430,363 (1.64 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 430,365 (1.64 MB)

In [38]:
best_model_3.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8907 - loss: 0.4194


[0.41943231225013733, 0.8906999826431274]

In [49]:
new_layers = []
feature_inxs = None
ratio_neurons_keep = 0.8
n_neurons_keep = int(243 * ratio_neurons_keep)

input_shape = best_model_3.input_shape[1:]
new_layers.append(tf.keras.layers.InputLayer(input_shape=input_shape))

for layer in best_model_3.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        w, b = layer.get_weights()

        if feature_inxs is not None:
            w = w[feature_inxs, :]

        w_sum = np.abs(w).sum(axis=0)
        inxs = np.sort(np.argsort(w_sum)[-n_neurons_keep:])
        feature_inxs = inxs

        new_w = w[:, inxs]
        new_b = b[inxs]

        new_dense = tf.keras.layers.Dense(len(inxs), input_dim=new_w.shape[0])
        new_dense.build((None, new_w.shape[0]))
        new_dense.set_weights([new_w, new_b])
        new_layers.append(new_dense)
        continue
    
    new_layers.append(layer)


reduced_neurons_model = tf.keras.Sequential(new_layers)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [50]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.0043925333029596605,
                                    momentum=0.95,
                                    nesterov=True)

reduced_neurons_model.compile(loss=best_model_3.loss, metrics=['accuracy'], optimizer=optimizer)
reduced_neurons_model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8614 - loss: 1.4591


[1.4591439962387085, 0.8614000082015991]

In [51]:
for layer in reduced_neurons_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        w, b = layer.get_weights()
        print(f'Layer {layer.name}', np.abs(w).sum())

Layer dense_12 9140.615
Layer dense_13 3277.5474
Layer dense_14 3227.6934
Layer dense_15 3168.9597
Layer dense_16 3071.8203
Layer dense_17 236.6404


In [52]:
reduced_neurons_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 194)            │       152,290 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 194)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 194)            │        37,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 194)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 194)            │        37,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 194)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 194)            │        37,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 194)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 194)            │        37,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 194)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 10)             │         1,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,560 (1.17 MB)

 Trainable params: 305,560 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

In [53]:
new_layers = reduced_neurons_model.layers[:6] + reduced_neurons_model.layers[8:]

reduced_layers_model = tf.keras.models.Sequential(new_layers)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.0043925333029596605,
                                    momentum=0.95,
                                    nesterov=True)

reduced_layers_model.compile(loss=best_model_3.loss, metrics=['accuracy'], optimizer=optimizer)
reduced_layers_model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.0770 - loss: 11.1020


[11.101974487304688, 0.07699999958276749]

In [59]:
new_layers = [
    tf.keras.layers.Input(X_train.shape[1:]),
    tf.keras.layers.Flatten(),
]

n_layers = len(best_model_3.layers)
factor = 10

for i, layer in enumerate(best_model_3.layers):
    if isinstance(layer, tf.keras.layers.Dense) and i < n_layers-1:
        units = layer.units - (factor * i)
        new_layer = tf.keras.layers.Dense(units, 
                                          activation=best_model_3.layers[i+1].activation, 
                                          kernel_initializer=layer.kernel_initializer)
        new_layers.append(new_layer)        


new_layers.append(tf.keras.layers.Dense(10, activation='softmax'))
final_model = tf.keras.models.Sequential(new_layers)

In [61]:
final_model.compile(loss='sparse_categorical_crossentropy', 
                    optimizer=tf.keras.optimizers.SGD(0.004392, momentum=0.95, nesterov=True),
                    metrics=['accuracy'])

final_model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid), 
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)])

Epoch 1/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - accuracy: 0.8658 - loss: 0.3612 - val_accuracy: 0.8662 - val_loss: 0.3659
Epoch 2/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.8791 - loss: 0.3276 - val_accuracy: 0.8706 - val_loss: 0.3555
Epoch 3/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.8874 - loss: 0.3041 - val_accuracy: 0.8740 - val_loss: 0.3333
Epoch 4/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.8928 - loss: 0.2874 - val_accuracy: 0.8714 - val_loss: 0.3501
Epoch 5/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.8966 - loss: 0.2763 - val_accuracy: 0.8740 - val_loss: 0.3499
Epoch 6/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.9003 - loss: 0.2621 - val_accuracy: 0.8802 - val_loss: 0.3309
Epoch 7/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9049 - loss: 0.2493 - val_accuracy: 0.8848 - val_loss: 0.3202
Epoch 8/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9101 -

In [62]:
final_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 233)            │       182,905 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 213)            │        49,842 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 193)            │        41,302 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 173)            │        33,562 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 153)            │        26,622 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 10)             │         1,540 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 671,548 (2.56 MB)

 Trainable params: 335,773 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 335,775 (1.28 MB)

In [65]:
def build_model2(hp):
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    optimizer_choice = hp.Choice('optimizer', ['sgd', 'adam', 'nadam', 'adamax'])
    match optimizer_choice:
      case 'nadam': optimizer = tf.keras.optimizers.Nadam(learning_rate)
      case 'adam': optimizer = tf.keras.optimizers.Adam(learning_rate)
      case 'adamax': optimizer = tf.keras.optimizers.Adamax(learning_rate)
      case 'sgd': optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                        momentum=hp.Choice("momentum", [0.0, 0.95]),
                                        nesterov=hp.Boolean("nesterov"))

    regularization = hp.Choice("regularization", ['BN', 'Dropout', ''])
    model = tf.keras.models.clone_model(final_model)

    if regularization:
        layers = []
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.Dense):
                match regularization:
                    case 'BN': layers.append(tf.keras.layers.BatchNormalization())
                    case 'Dropout': layers.append(tf.keras.layers.Dropout(rate=0.2))

            layers.append(layer)

        model = tf.keras.Sequential(layers)
    
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

In [66]:
tuner_2 = kt.RandomSearch(
    build_model2,
    objective="val_accuracy",
    max_trials=10,
    directory="fashion_mnist",
    project_name="params_final_2_111",
    seed=42,
)

tuner_2.search(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=5,
    batch_size=64,
)

Trial 10 Complete [00h 01m 42s]
val_accuracy: 0.88919997215271

Best val_accuracy So Far: 0.88919997215271
Total elapsed time: 00h 14m 01s


In [67]:
best_hp_2 = tuner_2.get_best_hyperparameters(1)[0]
best_hp_2.values

{'learning_rate': 0.00019584369813174676,
 'optimizer': 'adam',
 'momentum': 0.0,
 'nesterov': False,
 'regularization': 'BN'}

In [69]:
best_model_2 = tuner_2.get_best_models(1)[0]
best_model_2.fit(X_train, y_train, epochs=15, validation_data=(X_valid, y_valid),
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=5)])

Epoch 1/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 34s 16ms/step - accuracy: 0.8884 - loss: 0.3052 - val_accuracy: 0.8816 - val_loss: 0.3215
Epoch 2/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.8963 - loss: 0.2835 - val_accuracy: 0.8822 - val_loss: 0.3230
Epoch 3/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.9032 - loss: 0.2644 - val_accuracy: 0.8896 - val_loss: 0.3110
Epoch 4/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9080 - loss: 0.2510 - val_accuracy: 0.8880 - val_loss: 0.3096
Epoch 5/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 40s 16ms/step - accuracy: 0.9120 - loss: 0.2352 - val_accuracy: 0.8930 - val_loss: 0.2943
Epoch 6/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.9165 - loss: 0.2220 - val_accuracy: 0.8852 - val_loss: 0.2987
Epoch 7/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 30s 17ms/step - accuracy: 0.9216 - loss: 0.2109 - val_accuracy: 0.8932 - val_loss: 0.2963
Epoch 8/15
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9267 -

In [70]:
best_model_2.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8916 - loss: 0.3303


[0.3302903473377228, 0.8916000127792358]

In [71]:
best_model_2.save('final_model_2.keras')